<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v8`**
>
> Set the notebook kernel to *Python (mrigi_tor190_v8)* before running.

# Multi-Model Benchmarking: OPEN-ENDED Generation (No Context + MMR k=15)

Sister notebook to **23i**, but with **open-ended** questions instead of MCQ:

- Same 100 zeolite questions as 23i, but the model sees **only the question stem** — no A/B/C/D/E options and no letter-format instructions.
- Same 14 models (Llama-3-8B-Instruct + 13 fine-tunes).
- Same two retrieval methods: `no_context` and `mmr` (k=15).
- Answers are stored as free-form text; there is no correctness gate at generation time — grading is deferred to **30b** (GPT-4.1 judge).

In [2]:
import os
# NOTE: `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (used by 23i) requires
# torch >= 2.1. Kernel `mrigi_tor190_v8` has torch 1.12.1, which rejects the option
# with `Unrecognized CachingAllocator option: expandable_segments`, so we do NOT set it.
# We rely on the aggressive per-question / per-model cleanup instead.

import torch
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from collections import OrderedDict
import gc

from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load environment variables
with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            key, val = line[len('export '):].split('=', 1)
            os.environ[key] = val.strip('"').strip("'")

hf_token = os.environ.get("HF_TOKEN_BESTE")
if not hf_token:
    raise ValueError("HF_TOKEN_BESTE not found in env.sh")

os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print(f"✓ HuggingFace Hub token set (ending ...{hf_token[-4:]})")
print(f"✓ torch: {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")


✓ HuggingFace Hub token set (ending ...DOcr)
✓ torch: 1.12.1  |  CUDA available: True


In [3]:
!nvidia-smi

Mon Jul 20 20:26:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.119.02             Driver Version: 580.119.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:31:00.0 Off |                  Off |
| 30%   36C    P8             11W /  230W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
MODELS = OrderedDict([
    ('llama3_8b_instruct', {
        'name': 'meta-llama/Meta-Llama-3-8B-Instruct',
        'display_name': 'Llama-3-8B-Instruct',
        'gpu': 'cuda:0'
    }),
    ('dapt_lr1e5', {
        'name': 'aleynabeste/AllClassesAbstracts70Mmodel_LR1e5',
        'display_name': 'DAPT_LR1e5',
        'gpu': 'cuda:0'
    }),
    ('dapt_lr1e5_cot', {
        'name': 'aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT',
        'display_name': 'DAPT_LR1e5_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_step80', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_step80',
        'display_name': 'synv2V2_step80',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_step80_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_step80_COT_FT',
        'display_name': 'synv2V2_step80_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_final', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_final',
        'display_name': 'synv2V2_final',
        'gpu': 'cuda:0'
    }),
    ('synv2V2_final_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2V2_final_COT_FT',
        'display_name': 'synv2V2_final_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_step80', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_step80',
        'display_name': 'synv2_base_step80',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_step80_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_step80_COT_FT',
        'display_name': 'synv2_base_step80_COT',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_final', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_final',
        'display_name': 'synv2_base_final',
        'gpu': 'cuda:0'
    }),
    ('synv2_base_final_cot', {
        'name': 'aleynabeste/model_LR1e5v3_synv2_base_final_COT_FT',
        'display_name': 'synv2_base_final_COT',
        'gpu': 'cuda:0'
    }),
    ('fullpaper_120M', {
        'name': 'aleynabeste/model_LR1e5_fullpaper_longer_120M',
        'display_name': 'fullpaper_120M_LR1e5',
        'gpu': 'cuda:0'
    }),
    ('fullpaper_120M_cot', {
        'name': 'aleynabeste/model_LR1e5_fullpaper_longer_120M_COT_FT',
        'display_name': 'fullpaper_120M_COT',
        'gpu': 'cuda:0'
    }),
    ('base_llama_cot', {
        'name': 'aleynabeste/base_llama_intruct_COT_FT',
        'display_name': 'base_llama_COT',
        'gpu': 'cuda:0'
    }),
])

K_MMR = 15
MAX_NEW_TOKENS = 400  # 23i used 100 for MCQ; open-ended needs more room for a prose answer

# Cache the Llama-3-8B-Instruct chat template once so we can backfill it onto
# the DAPT / base fine-tunes whose tokenizers ship with chat_template=None.
_LLAMA3_CHAT_TEMPLATE = None

def _get_llama3_chat_template():
    global _LLAMA3_CHAT_TEMPLATE
    if _LLAMA3_CHAT_TEMPLATE is None:
        base_tok = AutoTokenizer.from_pretrained(
            'meta-llama/Meta-Llama-3-8B-Instruct',
            trust_remote_code=True, use_fast=True,
            token=hf_token, local_files_only=True,
        )
        _LLAMA3_CHAT_TEMPLATE = base_tok.chat_template
    return _LLAMA3_CHAT_TEMPLATE


def _gpu_mem_str(device='cuda:0'):
    a = torch.cuda.memory_allocated(device) / 1e9
    r = torch.cuda.memory_reserved(device) / 1e9
    return f"allocated={a:.2f}GB, reserved={r:.2f}GB"


def load_model(model_key):
    config = MODELS[model_key]
    model_name = config['name']
    device = config['gpu']

    print(f"\n{'='*80}")
    print(f"Loading {config['display_name']} on {device}...")
    print(f"  Before load: {_gpu_mem_str(device)}")
    print(f"{'='*80}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=True,
        token=hf_token,
        local_files_only=True
    )

    if getattr(tokenizer, 'chat_template', None) is None:
        tokenizer.chat_template = _get_llama3_chat_template()
        print(f"  ⚠ tokenizer.chat_template was None — backfilled from Llama-3-8B-Instruct")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        token=hf_token,
        local_files_only=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.1,
        do_sample=True,
    )
    llm = HuggingFacePipeline(pipeline=pipe)

    print(f"✓ {config['display_name']} loaded successfully on {device}")
    print(f"  After load:  {_gpu_mem_str(device)}")
    return llm, pipe, model, tokenizer


def cleanup_model(llm, pipe, model, tokenizer, device='cuda:0'):
    print(f"  Before cleanup: {_gpu_mem_str(device)}")
    try:
        model.to('cpu')
    except Exception as e:
        print(f"  (model.to('cpu') warn: {str(e)[:80]})")
    try:
        pipe.model = None
        pipe.tokenizer = None
    except Exception:
        pass
    try:
        llm.pipeline = None
    except Exception:
        pass
    del llm; del pipe; del model; del tokenizer
    for _ in range(3):
        gc.collect()
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print(f"  After cleanup:  {_gpu_mem_str(device)}")
    print("✓ GPU memory cleared")


print(f"✓ {len(MODELS)} models configured  (K_MMR={K_MMR}, max_new_tokens={MAX_NEW_TOKENS})")
for k, v in MODELS.items():
    print(f"  {v['display_name']:30s}  {v['name']}")

✓ 14 models configured  (K_MMR=15, max_new_tokens=400)
  Llama-3-8B-Instruct             meta-llama/Meta-Llama-3-8B-Instruct
  DAPT_LR1e5                      aleynabeste/AllClassesAbstracts70Mmodel_LR1e5
  DAPT_LR1e5_COT                  aleynabeste/AllClassesAbstracts70Mmodel_LR1e5_COT
  synv2V2_step80                  aleynabeste/model_LR1e5v3_synv2V2_step80
  synv2V2_step80_COT              aleynabeste/model_LR1e5v3_synv2V2_step80_COT_FT
  synv2V2_final                   aleynabeste/model_LR1e5v3_synv2V2_final
  synv2V2_final_COT               aleynabeste/model_LR1e5v3_synv2V2_final_COT_FT
  synv2_base_step80               aleynabeste/model_LR1e5v3_synv2_base_step80
  synv2_base_step80_COT           aleynabeste/model_LR1e5v3_synv2_base_step80_COT_FT
  synv2_base_final                aleynabeste/model_LR1e5v3_synv2_base_final
  synv2_base_final_COT            aleynabeste/model_LR1e5v3_synv2_base_final_COT_FT
  fullpaper_120M_LR1e5            aleynabeste/model_LR1e5_fullpaper_longer_

In [5]:
# ------------------------------------------------------------------
# Open-ended prompts — no A/B/C/D/E, no letter-format instruction.
# The model returns a free-text answer that the GPT-4.1 judge grades in 30b.
# ------------------------------------------------------------------
OPEN_ENDED_PROMPT = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using the provided context together with your own knowledge. Give a clear, focused answer in 3–5 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Context:
{context}

Question: {question}

Answer:"""

NO_CONTEXT_OPEN_ENDED_PROMPT = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using your own knowledge. Give a clear, focused answer in 3–5 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Question: {question}

Answer:"""


def extract_completion(full_response, tokenizer):
    """Isolate the assistant's response using Llama-3 special tokens."""
    assistant_tag = "<|start_header_id|>assistant<|end_header_id|>"
    if assistant_tag in full_response:
        completion = full_response.split(assistant_tag)[-1].strip()
    else:
        completion = full_response.strip()
    for token in ["<|eot_id|>", "<|end_of_text|>"]:
        completion = completion.replace(token, "").strip()
    return completion


def _clean_open_ended(text):
    """Light post-processing: strip a leading 'Answer:' if the model repeated it."""
    t = text.strip()
    for lead in ('Answer:', 'ANSWER:', 'answer:'):
        if t.startswith(lead):
            t = t[len(lead):].strip()
            break
    return t


def evaluate_openended_response(llm, query, correct_answer_text, correct_answer_letter,
                                 context, tokenizer, meta):
    """Generate an open-ended answer for a single question with context."""
    prompt_text = OPEN_ENDED_PROMPT.format(context=context, question=query)
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    try:
        full_response = llm(formatted_prompt)
        completion = extract_completion(full_response, tokenizer)
        answer = _clean_open_ended(completion)
        if not answer:
            return {
                'query': query, 'context': context,
                'model_answer': 'INVALID',
                'correct_answer_text': correct_answer_text,
                'correct_answer_letter': correct_answer_letter,
                'full_response': completion,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                'error': 'Empty completion',
            }
        return {
            'query': query, 'context': context,
            'model_answer': answer,
            'correct_answer_text': correct_answer_text,
            'correct_answer_letter': correct_answer_letter,
            'full_response': completion,
            'title': meta.get('title'), 'doi': meta.get('doi'),
        }
    except Exception as e:
        print(f"Processing error: {str(e)[:120]}")
        return {
            'query': query, 'context': context,
            'model_answer': 'ERROR',
            'correct_answer_text': correct_answer_text,
            'correct_answer_letter': correct_answer_letter,
            'title': meta.get('title'), 'doi': meta.get('doi'),
            'error': str(e),
        }


def evaluate_openended_no_context(llm, query, correct_answer_text, correct_answer_letter,
                                   tokenizer, meta):
    """Generate an open-ended answer for a single question without context."""
    prompt_text = NO_CONTEXT_OPEN_ENDED_PROMPT.format(question=query)
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    try:
        full_response = llm(formatted_prompt)
        completion = extract_completion(full_response, tokenizer)
        answer = _clean_open_ended(completion)
        if not answer:
            return {
                'query': query,
                'model_answer': 'INVALID',
                'correct_answer_text': correct_answer_text,
                'correct_answer_letter': correct_answer_letter,
                'full_response': completion,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                'error': 'Empty completion',
            }
        return {
            'query': query,
            'model_answer': answer,
            'correct_answer_text': correct_answer_text,
            'correct_answer_letter': correct_answer_letter,
            'full_response': completion,
            'title': meta.get('title'), 'doi': meta.get('doi'),
        }
    except Exception as e:
        print(f"Processing error: {str(e)[:120]}")
        return {
            'query': query,
            'model_answer': 'ERROR',
            'correct_answer_text': correct_answer_text,
            'correct_answer_letter': correct_answer_letter,
            'title': meta.get('title'), 'doi': meta.get('doi'),
            'error': str(e),
        }


def evaluate_method(llm, method_name, questions_df, tokenizer, k=15):
    """Run a single retrieval method over all questions. Clears GPU cache per question."""
    results = []
    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df),
                       desc=f"Evaluating {method_name} (k={k})"):
        query = row['Question']
        correct_text = row['Correct_Answer_Text']
        correct_letter = row['Correct_Answer_Letter']
        meta = {'title': row.get('Title'), 'doi': row.get('DOI')}
        try:
            if method_name == 'no_context':
                result = evaluate_openended_no_context(
                    llm, query, correct_text, correct_letter, tokenizer, meta
                )
            elif method_name == 'mmr':
                context = get_mmr_context(query, k=k)
                result = evaluate_openended_response(
                    llm, query, correct_text, correct_letter, context, tokenizer, meta
                )
            else:
                raise ValueError(f"Unknown method: {method_name}")
        except Exception as e:
            print(f"  ⚠ Error on question: {str(e)[:100]}")
            result = {
                'query': query, 'model_answer': 'ERROR',
                'correct_answer_text': correct_text,
                'correct_answer_letter': correct_letter,
                'title': meta.get('title'), 'doi': meta.get('doi'),
                'error': str(e),
            }
        results.append(result)
        gc.collect()
        torch.cuda.empty_cache()

    n_valid = sum(1 for r in results if r.get('model_answer') not in ('ERROR', 'INVALID'))
    total = len(results)
    valid_rate = (n_valid / total * 100) if total else 0
    return {
        'method': method_name, 'k': k,
        'n_valid': n_valid, 'total': total, 'valid_rate': valid_rate,
        'detailed_results': results,
    }

In [6]:
from pathlib import Path

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"

if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"✓ Loaded FAISS index with {vector_db.index.ntotal} vectors")


def get_mmr_context(query, k=15):
    results = vector_db.max_marginal_relevance_search(query, k=k)
    return "\n\n".join([doc.page_content for doc in results])


print("✓ MMR retrieval function defined")

/tmp/ipykernel_1596803/3570817700.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


✓ Loaded FAISS index with 1474439 vectors
✓ MMR retrieval function defined


In [7]:
# Load the 100 open-ended questions built from
# zeolite_mcq_selected_combined__modified_beste.xlsx (Selected MCQs minus 14 red rows,
# plus the first 14 rows of the replacements tab). See build script in the repo notes.
QUESTION_FILE = 'zeolite_openended_100.xlsx'

raw_df = pd.read_excel(QUESTION_FILE, engine='openpyxl')
raw_df = raw_df.dropna(subset=['question', 'correct_answer', 'correct_answer_text'])

questions_df = pd.DataFrame({
    'Question': raw_df['question'].astype(str).str.strip(),
    'Correct_Answer_Text': raw_df['correct_answer_text'].astype(str).str.strip(),
    'Correct_Answer_Letter': raw_df['correct_answer'].astype(str).str.strip().str.upper(),
    'Title': raw_df['title'].astype(str),
    'DOI': raw_df['doi'].astype(str),
}).reset_index(drop=True)

print(f"✓ Loaded {len(questions_df)} open-ended questions from {QUESTION_FILE}")
print(f"\nSample question 0:")
print(f"  Q: {questions_df.iloc[0]['Question'][:200]}")
print(f"  Gold ({questions_df.iloc[0]['Correct_Answer_Letter']}): {questions_df.iloc[0]['Correct_Answer_Text'][:200]}")

✓ Loaded 100 open-ended questions from zeolite_openended_100.xlsx

Sample question 0:
  Q: In zeolite synthesis from a sodium–organic–silica–alumina hydrogel, what is the most plausible broader role of an organic cation such as hexamethonium in steering whether ZSM-48, EU-1, or their co-cry
  Gold (A): It acts as a structure-directing agent whose fit and charge-compensation within different frameworks biases nucleation and growth toward specific zeolite topologies.


In [8]:
# Set RESUME=True to load an existing checkpoint and skip already-completed models.
# Set RESUME=False for a fresh run (checkpoint will be overwritten).
import time as _time

RESUME = True
CHECKPOINT_PATH = 'results_23j_checkpoint.json'
MAX_CONSECUTIVE_LOAD_FAILS = 2  # bail after this many back-to-back load failures

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_PATH = f'results_23j_complete_{timestamp}.json'
PROGRESS_LOG = f'results_23j_progress_{timestamp}.log'


def _is_model_complete(res, expected_total):
    """A model is 'done' only if both methods ran on exactly the current dataset
    size — protects against stale checkpoints from a different question set."""
    return (
        isinstance(res, dict)
        and 'no_context' in res and 'mmr' in res
        and res['no_context'].get('total', 0) == expected_total
        and res['mmr'].get('total', 0) == expected_total
    )


def _log(msg):
    """Print + append to a persistent progress log so long runs are auditable
    even if the notebook UI trims scrollback."""
    stamped = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(stamped, flush=True)
    with open(PROGRESS_LOG, 'a') as f:
        f.write(stamped + "\n")


def _save_checkpoint(all_results, tag):
    """Write both the persistent checkpoint and the timestamped complete file."""
    for path in (CHECKPOINT_PATH, SAVE_PATH):
        with open(path, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)
    sz = os.path.getsize(SAVE_PATH) / 1024
    _log(f"   💾 checkpoint saved [{tag}]: {SAVE_PATH} ({sz:.0f} KB)")


EXPECTED_TOTAL = len(questions_df)
_log(f"Expected questions per model: {EXPECTED_TOTAL}")
_log(f"Progress log: {PROGRESS_LOG}")

if RESUME and os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        all_results = json.load(f)
    done = [k for k, v in all_results.items() if _is_model_complete(v, EXPECTED_TOTAL)]
    stale = [k for k, v in all_results.items()
             if isinstance(v, dict) and not _is_model_complete(v, EXPECTED_TOTAL)
             and ('no_context' in v or 'mmr' in v or 'load_error' in v)]
    _log(f"✓ Resuming from {CHECKPOINT_PATH} — {len(done)} models already complete:")
    for d in done:
        _log(f"    ✓ {d}")
    if stale:
        _log(f"⚠ {len(stale)} models will be re-evaluated (stale, partial, or prior load_error):")
        for s in stale:
            _log(f"    ↻ {s}")
else:
    all_results = {}
    if RESUME:
        _log(f"No checkpoint found at {CHECKPOINT_PATH} — starting fresh.")
    else:
        _log(f"Fresh run — checkpoint {CHECKPOINT_PATH} will be overwritten.")

# ------------------------------------------------------------------
# Model loop
# ------------------------------------------------------------------
run_start = _time.time()
consecutive_load_fails = 0
model_items = list(MODELS.items())
n_total_models = len(model_items)

for m_idx, (model_key, model_config) in enumerate(model_items, start=1):
    display_name = model_config['display_name']

    if _is_model_complete(all_results.get(display_name), EXPECTED_TOTAL):
        _log(f"[{m_idx}/{n_total_models}] SKIP  {display_name} — already complete")
        continue

    model_start = _time.time()
    _log("")
    _log("#" * 80)
    _log(f"[{m_idx}/{n_total_models}] MODEL: {display_name}  (elapsed so far: {(model_start-run_start)/60:.1f} min)")
    _log("#" * 80)

    try:
        llm, pipe, model, tokenizer = load_model(model_key)
        consecutive_load_fails = 0  # reset on any successful load
    except Exception as e:
        consecutive_load_fails += 1
        _log(f"✗ Failed to load {display_name}: {str(e)[:200]}")
        all_results[display_name] = {'load_error': str(e)}
        _save_checkpoint(all_results, tag=f"load_fail:{display_name}")
        if consecutive_load_fails >= MAX_CONSECUTIVE_LOAD_FAILS:
            _log("")
            _log("!" * 80)
            _log(f"ABORTING: {consecutive_load_fails} consecutive load failures — this looks systemic")
            _log(f"(e.g. wrong kernel / torch flag / missing HF cache), not a per-model issue.")
            _log("!" * 80)
            break
        continue

    # Fresh dict for this model so a rerun doesn't keep stale per-method entries.
    all_results[display_name] = {}

    for method in ['no_context', 'mmr']:
        method_start = _time.time()
        _log(f"--- [{display_name}] {method}{' (k=' + str(K_MMR) + ')' if method == 'mmr' else ''} ---")
        try:
            result = evaluate_method(llm, method, questions_df, tokenizer, k=K_MMR)
            all_results[display_name][method] = result
            dt = (_time.time() - method_start) / 60
            _log(f"✓ {method}: {result['n_valid']}/{result['total']} valid  ({dt:.1f} min)")
        except Exception as e:
            _log(f"✗ {method} error: {str(e)[:200]}")
            all_results[display_name][method] = {
                'error': str(e), 'n_valid': 0, 'total': 0, 'valid_rate': 0,
            }
        # Save checkpoint after EACH method (not just after both) → less lost work on crash.
        _save_checkpoint(all_results, tag=f"{display_name}:{method}")

    cleanup_model(llm, pipe, model, tokenizer)
    dt_model = (_time.time() - model_start) / 60
    dt_total = (_time.time() - run_start) / 60
    _log(f"⏱  {display_name} done in {dt_model:.1f} min  (total: {dt_total:.1f} min)")

total_elapsed = (_time.time() - run_start) / 60
_log("")
_log("=" * 80)
_log(f"All models processed. Total wall-clock: {total_elapsed:.1f} min")
_log(f"Final checkpoint: {CHECKPOINT_PATH}")
_log(f"Timestamped file: {SAVE_PATH}")
_log(f"Progress log:     {PROGRESS_LOG}")
_log("=" * 80)


[20:26:47] Expected questions per model: 100
[20:26:47] Progress log: results_23j_progress_20260720_202647.log
[20:26:47] ✓ Resuming from results_23j_checkpoint.json — 0 models already complete:
[20:26:47] ⚠ 1 models will be re-evaluated (stale, partial, or prior load_error):
[20:26:47]     ↻ Llama-3-8B-Instruct
[20:26:47] 
[20:26:47] ################################################################################
[20:26:47] [1/14] MODEL: Llama-3-8B-Instruct  (elapsed so far: 0.0 min)
[20:26:47] ################################################################################

Loading Llama-3-8B-Instruct on cuda:0...
  Before load: allocated=0.00GB, reserved=0.00GB


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Llama-3-8B-Instruct loaded successfully on cuda:0
  After load:  allocated=16.19GB, reserved=16.20GB
[20:26:58] --- [Llama-3-8B-Instruct] no_context ---


/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/tmp/ipykernel_1596803/2694289414.py:138: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
Evaluating no_context (k=15):   0%|          | 0/100 [00:00<?, ?it/s]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/tmp/ipykernel_1596803/809578385

...

Evaluating mmr (k=15):   0%|          | 0/100 [00:00<?, ?it/s]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Evaluating mmr (k=15):   1%|          | 1/100 [00:05<09:35,  5.82s/it]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: Y

Processing error: CUDA out of memory. Tried to allocate 1.51 GiB (GPU 0; 23.55 GiB total capacity; 20.10 GiB already allocated; 1.04 GiB f


Evaluating mmr (k=15):  69%|██████▉   | 69/100 [05:44<02:32,  4.91s/it]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Evaluating mmr (k=15):  70%|███████   | 70/100 [05:49<02:31,  5.04s/it]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: User

[22:23:17] ✓ mmr: 99/100 valid  (8.4 min)


[22:23:18]    💾 checkpoint saved [fullpaper_120M_LR1e5:mmr]: results_23j_complete_20260720_202647.json (32438 KB)
  Before cleanup: allocated=16.29GB, reserved=16.30GB
  After cleanup:  allocated=0.09GB, reserved=0.11GB
✓ GPU memory cleared
[22:23:24] ⏱  fullpaper_120M_LR1e5 done in 15.1 min  (total: 116.6 min)
[22:23:24] 
[22:23:24] ################################################################################
[22:23:24] [13/14] MODEL: fullpaper_120M_COT  (elapsed so far: 116.6 min)
[22:23:24] ################################################################################

Loading fullpaper_120M_COT on cuda:0...
  Before load: allocated=0.09GB, reserved=0.11GB
[22:23:24] ✗ Failed to load fullpaper_120M_COT: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like aleynabeste/model_LR1e5_fullpaper_longer_120M_COT_FT is not the path to a dire
[22:23:24]    💾 checkpoint saved [load_fail:fullpaper_120M_COT]: results_23j_c

In [9]:
# Quick sanity summary: how many valid (non-ERROR/INVALID) answers per model per method?
# The real quality signal comes from GPT-4.1 judge in notebook 30b.
print("\n" + "="*90)
print(f"RESULTS SUMMARY: Open-Ended, No Context vs MMR (k={K_MMR}) — n={EXPECTED_TOTAL}")
print("="*90)
print(f"{'Model':<35} {'no_context (valid)':>22} {'mmr_k15 (valid)':>22}")
print("-"*90)

for display_name, res in all_results.items():
    if 'load_error' in res:
        print(f"{display_name:<35} {'LOAD ERROR':>22}")
        continue
    nc = res.get('no_context', {})
    mm = res.get('mmr', {})
    nc_str = f"{nc.get('n_valid', 0)}/{nc.get('total', 0)}"
    mm_str = f"{mm.get('n_valid', 0)}/{mm.get('total', 0)}"
    print(f"{display_name:<35} {nc_str:>22} {mm_str:>22}")

print("="*90)
print("(Note: 'valid' just means the model returned a non-empty answer.")
print(" Actual quality scores come from GPT-4.1 in 30b.)")


RESULTS SUMMARY: Open-Ended, No Context vs MMR (k=15) — n=100
Model                                   no_context (valid)        mmr_k15 (valid)
------------------------------------------------------------------------------------------
Llama-3-8B-Instruct                                100/100                 99/100
DAPT_LR1e5                                         100/100                 99/100
DAPT_LR1e5_COT                                  LOAD ERROR
synv2V2_step80                                     100/100                 99/100
synv2V2_step80_COT                              LOAD ERROR
synv2V2_final                                      100/100                 99/100
synv2V2_final_COT                               LOAD ERROR
synv2_base_step80                                  100/100                 99/100
synv2_base_step80_COT                           LOAD ERROR
synv2_base_final                                   100/100                 99/100
synv2_base_final_COT                

In [10]:
# CSV of valid-response rates per model x method (a quick health summary,
# not a quality score — those come from 30b).
rows = []
for display_name, res in all_results.items():
    if 'load_error' in res:
        continue
    for method in ['no_context', 'mmr']:
        r = res.get(method, {})
        rows.append({
            'model': display_name,
            'method': method,
            'k': K_MMR if method == 'mmr' else 'n/a',
            'n_valid': r.get('n_valid', 0),
            'total': r.get('total', 0),
            'valid_rate_pct': r.get('valid_rate', 0),
        })

csv_df = pd.DataFrame(rows)
csv_path = f'results_23j_{timestamp}.csv'
csv_df.to_csv(csv_path, index=False)
print(f"✓ CSV saved: {csv_path}")
print(csv_df.to_string(index=False))

✓ CSV saved: results_23j_20260720_202647.csv
               model     method   k  n_valid  total  valid_rate_pct
 Llama-3-8B-Instruct no_context n/a      100    100           100.0
 Llama-3-8B-Instruct        mmr  15       99    100            99.0
          DAPT_LR1e5 no_context n/a      100    100           100.0
          DAPT_LR1e5        mmr  15       99    100            99.0
      synv2V2_step80 no_context n/a      100    100           100.0
      synv2V2_step80        mmr  15       99    100            99.0
       synv2V2_final no_context n/a      100    100           100.0
       synv2V2_final        mmr  15       99    100            99.0
   synv2_base_step80 no_context n/a      100    100           100.0
   synv2_base_step80        mmr  15       99    100            99.0
    synv2_base_final no_context n/a      100    100           100.0
    synv2_base_final        mmr  15       99    100            99.0
fullpaper_120M_LR1e5 no_context n/a      100    100           100.0
ful

In [11]:
# Persist the full detailed results (used by 30b as input).
final_path = f'results_23j_complete_{timestamp}.json'
with open(final_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"✓ Final results saved: {final_path} ({size_mb:.1f} MB)")

total_saved = sum(
    len(res.get(m, {}).get('detailed_results', []))
    for res in all_results.values()
    for m in ['no_context', 'mmr']
    if 'load_error' not in res
)
print(f"  Total question-level records: {total_saved}")

✓ Final results saved: results_23j_complete_20260720_202647.json (31.7 MB)
  Total question-level records: 1400
